In [1]:
import random           #create random numbers for the synthetic data
import re               #regular expression: find and replace patterns in text
from pathlib import Path  #handles file and folder paths on any operating system


import pandas as pd     #tables (Data Frame) and data cleaning
import plotly.express as px  #quick intractive charts

random.seed(7)           #fixes the random numbers so every run given the same data
pd.set_option("display.max_colwidth", 120)  #show long ticket without cutting it off

In [2]:
DATA_DIR = Path("data")          #name of the folder that stores our files
RAW_PATH = DATA_DIR / "tickets_raw.csv"   #path of the untouched, raw dataset
CLEAN_PATH = DATA_DIR / "tickets_clean.csv"   #path of the cleaned dataset
DATA_DIR.mkdir(exist_ok=True)      #create the folder; no error if it already exists

In [3]:
####BUILLDING BLOCKS FOR SYNTHETIC TICKETS

In [3]:
OPENERS = {
    "Negative": ["I am really frustrated.",     #first sentence of  a ticket, grouped by sentiment
                 "This is unacceptable.",       #sentence that sound angry or upset
                 "I'm very disappointed.",
                 "I am extremely upset.",
                 "This is the wrost experience."],
    
    "Neutral": ["Hello," "Hi team,",             #sentence with no strong emotion
                "Good morning," "I have a question.",
                "Please check this for me."],
    
    "Positive": ["Thanks for the great service so far.",   #sentence that sound happy or polite
                 "I love your product, but",
                 "Great app overall, however"],
}

URGENCY = ["Please fix this urgently.",          #phrase that signal urgency
           "I need this resolved immediately.",  #used later in the priority rules
           "This is an emergency.",
           "Pespond ASAP."]

CORE = {                             #main problem sentence, grouped by category
    "Billing and Payments": [         #category 1: bills and payments
        "I was charged twice for my subscription this month."
        "There is an incorrect amount on my latest invoice.",
        "My credit card payment failed but money was deducted.",
        "I cannot find my invoice for order {n}.",     #{n} is replaced by a random oder number
        "My payment is showing as pending for three days.",
        "An unknown charge appeared on my statement.",
    ],

    "Technical Issues": [                #category 2: app and website faults
        "The app crashes every time I open the dashboard.",
        "The website shows an error 500 when I submit the form.",
        "Pages are loading very slowly since the last update."
        "I cannot uploading files, the upload bar gets stuck.",
        "Notifications are not working on my phone.",
        "The software kepps disconnecting from the server,",
    ],

    "Account Access": [                  #category 3: login and account problems
        "I cannot log in to my account, the password is not accepted.",
        "i forgot my password and the reset email never arrives.",
        "My account has been locked after several login attempts.",
        "The verification code for two factor login is not working.",
        "My account was hacked and I cannot access it.",
        "The login page kepps redirecting me back to itself.",
    ],

    "Refund Requests": [              #category 4: refund requests
        "I would like a refund for order {n}.",
        "please refund my last paymnet, I cancelled the service.",
        "I returned the item two weeks ago and still have no refund.",
        "I wnat my money back for the annual plan I never used.",
        "The refund ypu promised has not been credited yet.",
        "Can I get a refund for the duplicate purchase?",
    ],

    "Product or Service Complaints": [          #category 5: complaints about products or service 
        "The product quality is much wrose than advertised.",
        "Your delivery arrived late and the box was damaged.",
        "The customer service agent was rude to me yesterday.",
        "The item I received is defective and stopped working.",
        "I waited an hour on the phone nobody helped me.",
        "The packaging was open and parts were missing,",
    ],

}

print("Categories:", list(CORE.keys()))      #shows the 5 category names

    
    

    



Categories: ['Billing and Payments', 'Technical Issues', 'Account Access', 'Refund Requests', 'Product or Service Complaints']


In [5]:
## FUNCTION THAT GENERATE THE DATASET

In [4]:
def generate_synthetic(n_rows=2000):         # n_rows = how many tickets to create (default 2000)
    rows = []                                # each finished ticket is stored in this list
    for i in range(n_rows):                  # repeat once per ticket
        category = random.choice(list(CORE)) # pick a random category
        sentiment = random.choices(          # pick a random sentiment with different probabilities
            list(OPENERS), weights=[0.5, 0.3, 0.2])[0]   # 50% negative, 30% Neutral, 20% Positive

        parts = [random.choice(OPENERS[sentiment]),   # openers sentence that matches the sentiment
                 random.choice(CORE[category]).format(# problem sentence that matches the category
                     n=random.randint(10000, 99999))]  # fill {n} with random order number
        if random.random() < 0.35:          # in about 35% of tickets...
            parts.append(random.choice(URGENCY))   # ...add an urgency phrase

        rows.append({"ticket_id": f"T{i + 1:05d}",       # ID such as T00001, T00002
                     "text": " ".join(parts),            # join the sentence into one ticket text
                     "category": category,               # true category label
                     "sentiment": sentiment})             # true sentiment label

    df = pd.DataFrame(rows)                # convert the list of tickets into a table

    #-- add real world mess so there is something to clean --

    df = pd.concat([df, df.sample(40, random_state=1)],   #copy 40 random tickets and append them
                   ignore_index=True)                     # this creates duplicate records
    df.loc[df.sample(15, random_state=2).index, "text"] = None    # blank out the text of 15 tickets ( missing values)
    messy = df.sample(10, random_state=3).index          # choose 10 random rows
    df.loc[messy, "text"] = df.loc[messy, "text"].fillna("") + \
        "   Contact me sree@example.com or visit https://example.com!!!   "     # add an email, URL and emoki
    return df                         # gives the finished table back
                     
        

In [7]:
## CREATE AND SAVE THE RAW DATASET

In [5]:
raw = generate_synthetic(2000)        # build the dataset
raw.to_csv(RAW_PATH, index=False)     # save as CSV without the row -number column
print(f"Saved {len(raw)} raw tickets to {RAW_PATH}")  # confirm how many rows were saved

Saved 2040 raw tickets to data\tickets_raw.csv


In [9]:
# LOAD AND VIEW THE DATA

In [6]:
df = pd.read_csv(RAW_PATH)       # read the CSV file into a DataFrame
df.head(8)             # display the first 9 rows

,ticket_id,text,category,sentiment
0,T00001,"I love your product, but The login page kepps redirecting me back to itself. Please fix this urgently.",Account Access,Positive
1,T00002,Please check this for me. i forgot my password and the reset email never arrives. Pespond ASAP.,Account Access,Neutral
2,T00003,This is the wrost experience. My payment is showing as pending for three days.,Billing and Payments,Negative
3,T00004,"Great app overall, however An unknown charge appeared on my statement.",Billing and Payments,Positive
4,T00005,This is unacceptable. I would like a refund for order 82963.,Refund Requests,Negative
5,T00006,"This is the wrost experience. I cannot log in to my account, the password is not accepted. I need this resolved imme...",Account Access,Negative
6,T00007,Please check this for me. My credit card payment failed but money was deducted. Please fix this urgently.,Billing and Payments,Neutral
7,T00008,This is unacceptable. The item I received is defective and stopped working.,Product or Service Complaints,Negative


In [11]:
## SHAPE, COLUMN AND DATA TYPES

In [7]:
print("Shape (rows, columns):", df.shape)   # number of rows and columns 
print()                                     # print an empty line for spacing
df.info()                                   # column names, data types and non-null counts

Shape (rows, columns): (2040, 4)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2040 entries, 0 to 2039
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ticket_id  2040 non-null   object
 1   text       2025 non-null   object
 2   category   2040 non-null   object
 3   sentiment  2040 non-null   object
dtypes: object(4)
memory usage: 63.9+ KB


In [13]:
print("Missing values per column:")    # heading
print(df.isna().sum())           # count of empty values in each column
print()                          # empty line
print("Duplicate tickets texts :",
      df.duplicated(subset=["text"]).sum())  # rows whose text repeats an earlier row
print("Blank texts            :",
       (df["text"].fillna("").str.strip() == "").sum())     # text that contain only  spaces or nothing

Missing values per column:
ticket_id     0
text         15
category      0
sentiment     0
dtype: int64

Duplicate tickets texts : 1076
Blank texts            : 15


In [14]:
## CATEGORY DISTRIBUTION

In [8]:
cat_counts = df["category"].value_counts().reset_index()    # number of tickets in each category
cat_counts.columns = ["category", "tickets"]         # gives the two columns clear names
display(cat_counts)                                 #shows the table (works in jupyter)

fig = px.bar(cat_counts, x="tickets", y="category",
             orientation="h",           # horizontal bars
             title="Tickets per category (raw data)")    # chart title
fig.update_layout(yaxis={"categoryorder": "total ascending"})   # sort bar by size
fig.show()              # draw the chart

,category,tickets
0,Account Access,424
1,Billing and Payments,423
2,Technical Issues,409
3,Refund Requests,394
4,Product or Service Complaints,390


In [16]:
## SENTIMENT DISTRIBUTION

In [9]:
sent_counts = df["sentiment"].value_counts().reset_index()   # number of tickets per sentiment
sent_counts.columns = ["sentiment", "tickets"]  # name the columns

px.bar(sent_counts, x="sentiment", y="tickets", color="sentiment",    # one color per sentiment
       color_discrete_map={"Negative": "#e5484d",     # Negative = red
                           "Neutral": "#8b8d98",       # Neutral = grey
                           "Positive": "#30a46c"},     # Positive = green
       title="Ticket per sentiment (raw data)").show() #ass title and draw the chart
                           

In [18]:
## TICKET LENGTH

In [10]:
word_counts = df["text"].dropna().str.split().str.len()   # number of words in each ticket
print(word_counts.describe().round(1))          # min, max, mean, and quartiles

px.histogram(word_counts, nbins=20,       # histogram with 20 bars
             title="Words per ticket (raw data)",
             labels={"value": "words"}      # x-axis label
            ).update_layout(showlegend=False).show()  # hide the legend and draw the chart

count    2025.0
mean       15.4
std         3.4
min         9.0
25%        13.0
50%        15.0
75%        17.0
max        29.0
Name: text, dtype: float64


In [20]:
## HANDLE MISSING VALUES

In [11]:
data = df.copy()       # work on a copy so the orginal stays unchanged
before = len(data)     # row count before cleaning

data = data.dropna(subset=["text"])     # remove rows where text is missing
data = data[data["text"].astype(str).str.strip() !=""]   # remove rows where text is only spaces

print(f"Rows before: {before}")     # rows before cleaning
print(f"Rows after : {len(data)}")  # rows after cleaning
print(f"Removed    : {before - len(data)} rows with missing/blank text")   # how many were removed

Rows before: 2040
Rows after : 2025
Removed    : 15 rows with missing/blank text


In [22]:
## TEXT CLEANING FUNCTION

In [12]:
URL_RE      = re.compile(r"https?://\S+|www\.\S+")  # matches web links (htto://, https://, www.)
EMAIL_RE    = re.compile(r"\S+@\S+")                # matches email address such as abc@xyz.com
NON_TEXT_RE = re.compile(r"[^a-z0-9\s']")           # matches anything except letters, digits, spaces, and  apostrophes
SPACE_RE    = re.compile(r"\s+")                    # matches one or more whitespace characters

def clean_text(text):                           # cleans a single ticket text
    if not isinstance(text, str):               # if the value is not text (for example NaN)...
        return ""                               # ... return an empty string
    text = text.lower()                         # lowercase everything: "Refund" becomes "refund"
    text = URL_RE.sub(" ", text)                # replace web links with a space
    text = EMAIL_RE.sub(" ", text)              # replace email adddresses with a space
    text = NON_TEXT_RE.sub(" ", text)           # replace symbols, punctuation and emojis with a space
    return SPACE_RE.sub(" ", text).strip()      # collapse repeated spaces and train both ends
        

In [24]:
## TEST THE CLEANING FUNCTION

In [13]:
examples = [                      # three test sentences
    "I CAN'T log in!!!   Contact me at sree@example.com",
    "Visit https://example.com for details... Urgent 😡",
    "Refund for ORDER #48213 please!!",
]

for e in examples:              # go through them one by one
    print("RAW   :", e)         # the orginal text
    print("CLEAN:", clean_text(e))   # the cleaned text
    print()                       # empty line between examples

RAW   : I CAN'T log in!!!   Contact me at sree@example.com
CLEAN: i can't log in contact me at

RAW   : Visit https://example.com for details... Urgent 😡
CLEAN: visit for details urgent

RAW   : Refund for ORDER #48213 please!!
CLEAN: refund for order 48213 please



In [26]:
## APPLY CLEANING TO EVERY TICKET

In [14]:
data["clean_text"] = data["text"].apply(clean_text)  # clean every ticket and store it in a new column
data = data[data["clean_text"] != ""]

data[["text", "clean_text"]].sample(5, random_state=4)  #show 5 random before/after examples

,text,clean_text
306,Please check this for me. An unknown charge appeared on my statement.,please check this for me an unknown charge appeared on my statement
1799,"Hello,Hi team, The app crashes every time I open the dashboard.",hello hi team the app crashes every time i open the dashboard
354,"I'm very disappointed. The packaging was open and parts were missing,",i'm very disappointed the packaging was open and parts were missing
635,Please check this for me. I was charged twice for my subscription this month.There is an incorrect amount on my late...,please check this for me i was charged twice for my subscription this month there is an incorrect amount on my lates...
225,I am really frustrated. My account has been locked after several login attempts.,i am really frustrated my account has been locked after several login attempts


In [28]:
## REMOVE DUPLICATE TICKETS

In [15]:
n_before = len(data)                        # row count before removing duplicates
data = data.drop_duplicates(subset=["clean_text"])    # keep the first copy of each identical tickets
print(f"Duplicates removed: {n_before - len(data)}")  # how many duplicates were removed
print(f"Tickets remaining : {len(data)}")   #how many tickets are left


Duplicates removed: 1062
Tickets remaining : 963


In [30]:
## PREPARE THE DATASET FOR MODEL TRAINING

In [16]:
data["category"]  = data["category"].astype(str).str.strip()   #remove stray spaces from category labels
data["sentiment"] = data["sentiment"].astype(str).str.strip().str.title()  # "negative become "Negative"
data["word_count"] = data["clean_text"].str.split().str.len()      # number of words in each cleaned ticket

data = data.reset_index(drop=True)          # renumber rows 0, 1, 2... after removals
data.head()                     # show the first 5 rows

,ticket_id,text,category,sentiment,clean_text,word_count
0,T00001,"I love your product, but The login page kepps redirecting me back to itself. Please fix this urgently.",Account Access,Positive,i love your product but the login page kepps redirecting me back to itself please fix this urgently,18
1,T00002,Please check this for me. i forgot my password and the reset email never arrives. Pespond ASAP.,Account Access,Neutral,please check this for me i forgot my password and the reset email never arrives pespond asap,17
2,T00003,This is the wrost experience. My payment is showing as pending for three days.,Billing and Payments,Negative,this is the wrost experience my payment is showing as pending for three days,14
3,T00004,"Great app overall, however An unknown charge appeared on my statement.",Billing and Payments,Positive,great app overall however an unknown charge appeared on my statement,11
4,T00005,This is unacceptable. I would like a refund for order 82963.,Refund Requests,Negative,this is unacceptable i would like a refund for order 82963,11


In [32]:
## BEFORE VS AFTER CHECK

In [17]:
summary = pd.DataFrame({              # comparison table
    "Raw data": [len(df),             # rows in the raw data
                 df["text"].isna().sum(),      #missing texts in the raw data
                 df.duplicated(subset=["text"]).sum()],    # duplicates in the raw data
    "Cleaned data": [len(data),                         # rows  in the cleaned data
                     data["text"].isna().sum(),     # missing texts (should be 0)
                     data.duplicated(subset=["clean_text"]).sum()],   # duplicates should be 0
}, index=["Rows", "Missing text", "Duplicate tickets"])       # rows labels of the table
summary                                                 # display the table


,Raw data,Cleaned data
Rows,2040,963
Missing text,15,0
Duplicate tickets,1076,0


In [34]:
## CLASS BALANCE AFTER CLEANING

In [18]:
print("Category balance after cleaning:")   # heading
print(data["category"].value_counts())      # tickets per category
print()                                     # empty line
print("Sentiment balance after cleaning:")  # heading
print(data["sentiment"].value_counts())     # tickets per sentiment

Category balance after cleaning:
category
Billing and Payments             227
Refund Requests                  218
Product or Service Complaints    182
Account Access                   180
Technical Issues                 156
Name: count, dtype: int64

Sentiment balance after cleaning:
sentiment
Negative    470
Neutral     265
Positive    228
Name: count, dtype: int64


In [36]:
## Lemmatisation cell( NLTK's WordNetLemmatizer )

In [19]:
import nltk                                     # NLTK: Natural Language Toolkit
from nltk.stem import WordNetLemmatizer         # reduces words to their base form
from nltk.corpus import stopwords               # list of common words such as "the", "is", "and"

nltk.download("wordnet", quiet=True)            # download the dictionary used by the lemmatizer
nltk.download("stopwords", quiet=True)          # download the stop word list

lemmatizer = WordNetLemmatizer()                # create the lemmatizer object
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "never"}   # stop words, but keep negations


def lemmatize_text(text):                       # normalises one cleaned ticket
    return " ".join(                            # join the remaining words back into a sentence
        lemmatizer.lemmatize(w)                 # reduce each word to its base form
        for w in text.split()                   # go through the words one by one
        if w not in stop_words)                 # skip stop words


data["lemma_text"] = data["clean_text"].apply(lemmatize_text)   # apply to every ticket
data[["clean_text", "lemma_text"]].head(3)      # show before/after examples

,clean_text,lemma_text
0,i love your product but the login page kepps redirecting me back to itself please fix this urgently,love product login page kepps redirecting back please fix urgently
1,please check this for me i forgot my password and the reset email never arrives pespond asap,please check forgot password reset email never arrives pespond asap
2,this is the wrost experience my payment is showing as pending for three days,wrost experience payment showing pending three day


In [38]:
## SAVE THE CLEAN DATASET

In [20]:
data.to_csv(CLEAN_PATH, index=False)    # save the cleaned dataset
print(f"Saved {len(data)} clean tickets to {CLEAN_PATH}")  # how many tickets were saved
print("Columns:", list(data.columns))         # list of final columns

Saved 963 clean tickets to data\tickets_clean.csv
Columns: ['ticket_id', 'text', 'category', 'sentiment', 'clean_text', 'word_count', 'lemma_text']


In [40]:
##MODEL TRAINING

In [21]:
from sklearn.model_selection import train_test_split   # scikit-learn's splitting function

train_df, test_df = train_test_split(           # split the table into two tables
    data,                                       # the cleaned dataset
    test_size=0.2,                              # 20% of tickets go to the test set
    stratify=data["category"],                  # keep the same category proportions in both sets
    random_state=42)                            # fixed seed so the split is repeatable

train_df.to_csv(DATA_DIR / "train.csv", index=False)   # save the training set
test_df.to_csv(DATA_DIR / "test.csv", index=False)     # save the test set

print("Training tickets:", len(train_df))       # size of the training set
print("Test tickets    :", len(test_df))        # size of the test set
print("Tickets in both sets:",                  # leakage check
      len(set(train_df["clean_text"]) & set(test_df["clean_text"])))   # must be 0

Training tickets: 770
Test tickets    : 193
Tickets in both sets: 0


In [42]:
##FINAL_CHECK

In [22]:
expected_categories = {"Billing and Payments", "Technical Issues", "Account Access",
                       "Refund Requests", "Product or Service Complaints"}

train_chk = pd.read_csv(DATA_DIR / "train.csv")          # re-read the saved files from disk
test_chk = pd.read_csv(DATA_DIR / "test.csv")

checks = {                                               # each check is True (pass) or False (fail)
    "Category names match the brief":
        set(train_chk["category"]) == expected_categories,
    "No missing text in train/test":
        train_chk["text"].notna().all() and test_chk["text"].notna().all(),
    "No duplicate tickets in train":
        not train_chk.duplicated(subset=["clean_text"]).any(),
    "No ticket appears in both train and test":
        len(set(train_chk["clean_text"]) & set(test_chk["clean_text"])) == 0,
    "lemma_text column exists":
        "lemma_text" in train_chk.columns and "lemma_text" in test_chk.columns,
    "Split is about 80/20":
        0.78 <= len(train_chk) / (len(train_chk) + len(test_chk)) <= 0.82,
    "All 4 files exist":
        all((DATA_DIR / f).exists() for f in ["tickets_raw.csv", "tickets_clean.csv", "train.csv", "test.csv"]),
}

for name, ok in checks.items():                          # print one line per check
    print(("PASS  " if ok else "FAIL  ") + name)
print("\nModule 1 is COMPLETE" if all(checks.values()) else "\nFix the FAIL items above")

PASS  Category names match the brief
PASS  No missing text in train/test
PASS  No duplicate tickets in train
PASS  No ticket appears in both train and test
PASS  lemma_text column exists
PASS  Split is about 80/20
PASS  All 4 files exist

Module 1 is COMPLETE


In [44]:
## MODULE - 2

In [45]:
## IMPORT LIBRARIES


In [23]:
import re                  # regular expression for text cleaning
from pathlib import Path   # handles files and folder paths

import joblib                    # SAVE AND LOADS TRAINED MODELS
import numpy as np               # numerical array
import pandas as pd              # tables (Data Frame)
import plotly.express as px      # interactive charts

from sklearn.dummy import DummyClassifier    # simple baseline model for comparison
from sklearn.feature_extraction.text import TfidfVectorizer    # convert text into TF - IDF numbers
from sklearn.linear_model import LogisticRegression    # the classification model
from sklearn.metrics import (accuracy_score,           # overall share of correct prediction
                             classification_report,    # precision, recall, F1 for each class
                             confusion_matrix)         # table of correct and wrong predictions
from sklearn.model_selection import StratifiedKFold, cross_val_score    # crosss-validation tools
from sklearn.pipeline import Pipeline                   # chains TF-IDF and classifier together

pd.set_option("display.max_colwidth", 120)              # show long ticket without cutting it off

In [47]:
## LOAD THE TRAINING AND TEST DATA

In [24]:
DATA_DIR = Path("data")           # folder created in Module 1
MODEL_DIR = Path("models")        # folder where we will save the trained model
MODEL_DIR.mkdir(exist_ok=True)    # create it if it does not exist

train_df = pd.read_csv(DATA_DIR / "train.csv")    # training tickets
train_df = pd.read_csv(DATA_DIR / "test.csv")       # test tickets

print("Training tickets:", len(train_df))           # size of the training set
print("Test tickets    :", len(test_df))            # size of the test set
train_df[["clean_text", "category"]].head()         # preview the text and the label

Training tickets: 193
Test tickets    : 193


,clean_text,category
0,good morning i have a question i would like a refund for order 31832,Refund Requests
1,please check this for me i waited an hour on the phone nobody helped me,Product or Service Complaints
2,i love your product but the software kepps disconnecting from the server pespond asap,Technical Issues
3,great app overall however i was charged twice for my subscription this month there is an incorrect amount on my late...,Billing and Payments
4,great app overall however i forgot my password and the reset email never arrives this is an emergency,Account Access


In [49]:
## CHOOSE INPUT AND TARGET COLUMN

In [25]:
TEXT_COLUMN = "clean_text"           # which text column to learn from
TARGET_COLUMN = "category"           # the label we want to predict

X_train = train_df[TEXT_COLUMN]      # TRAINING INPUT: TICKET
y_train = train_df[TARGET_COLUMN]    # training target: true category
X_test = test_df[TEXT_COLUMN]        # test input: ticket text
y_test = test_df[TARGET_COLUMN]      # test target: true category

labels = sorted(y_train.unique())    # the list of the category names, in a fixed order
print("Categories:", labels)         # show the categories

Categories: ['Account Access', 'Billing and Payments', 'Product or Service Complaints', 'Refund Requests', 'Technical Issues']


In [51]:
## BASELINE_MODELS

In [26]:
baseline = DummyClassifier(strategy="most_frequent")      # always predict the most frequent category
baseline.fit(X_train, y_train)                            # "train" it (it only learns the most common labels)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))    # accuracy on the test set
print(f"Baseline accuracy: {baseline_acc: .1%}")         # expected to be around 20% for 5 balanced classes

Baseline accuracy:  23.8%


In [53]:
## TF-IDF TEXT REPRESENTATION

In [27]:
tfidf = TfidfVectorizer(   # create the TF-IDF converter       
    ngram_range=(1, 2),    # single words and two-word phrases
    stop_words="english",  # remove common English words
    min_df=2,              # ignore very rare words
    sublinear_tf=True      # dampen the effect of repeated words
)

X_train_tfidf = tfidf.fit_transform(X_train)     # learn the vocabulary from TRAINING data and convert it
X_test_tfidf = tfidf.transform(X_test)           # convert test data with the SAME vocabulary (no re-learning)


print("Training matrix shape:", X_train_tfidf.shape)     # (tickets, number of features)
print("Testing matrix shape:", X_test_tfidf.shape)       # same number of features as training

Training matrix shape: (193, 321)
Testing matrix shape: (193, 321)


In [55]:
## TF-IDF learned

In [28]:
feature_names = np.array(tfidf.get_feature_names_out())       # all words and phrases the vectorizer knows
print("Vocabulary size:", len(feature_names))                # how many feature there are
print("Sample feature:", list(feature_names[:5]))            # first 15 feature

row = X_train_tfidf[0].toarray()[0]                          # TF - IDF values of the first training ticket
top_idx = row.argsort()[::-1][:6]                            # positions of its 6 highest scores
print("\nExample ticket:", X_train.iloc[0])                   # the ticket text
print("Top TF-IDF terms:", [(feature_names[i], round(float(row[i]), 3)) for i in top_idx])   # words with score

Vocabulary size: 321
Sample feature: ['500', '500 submit', 'accepted', 'access', 'access emergency']

Example ticket: good morning i have a question i would like a refund for order 31832
Top TF-IDF terms: [('question like', 0.462), ('refund order', 0.302), ('like refund', 0.302), ('like', 0.302), ('question', 0.284), ('morning', 0.284)]


In [57]:
## TRAIN LOGISTIC 

In [29]:
model = LogisticRegression(                #create the classifier
    max_iter=1000,                         # maximum training iteration
    class_weight="balanced",               # treat all categories fairly
    random_state=42)                       # fied seed for repeatable result

model.fit(X_train_tfidf, y_train)          # learn from the training tickets
print("Model trained. Classes:", list(model.classes_))    # confirm training and show the classes

Model trained. Classes: ['Account Access', 'Billing and Payments', 'Product or Service Complaints', 'Refund Requests', 'Technical Issues']


In [34]:
y_pred = model.predict(X_test_tfidf)      # predicted category for each test ticket

results = test_df[["text"]].copy()          # start a result table with the orginal ticket text
results["actual"] = y_test.values           # true category
results["predicted"] = y_pred               # predicted category
results["correct"] = results["actual"] == results["predicted"]   # True when the prediction is right
results.head(8)

,text,actual,predicted,correct
889,"Good morning,I have a question. I would like a refund for order 31832.",Refund Requests,Refund Requests,True
744,Please check this for me. I waited an hour on the phone nobody helped me.,Product or Service Complaints,Product or Service Complaints,True
451,"I love your product, but The software kepps disconnecting from the server, Pespond ASAP.",Technical Issues,Technical Issues,True
911,"Great app overall, however I was charged twice for my subscription this month.There is an incorrect amount on my lat...",Billing and Payments,Billing and Payments,True
279,"Great app overall, however i forgot my password and the reset email never arrives. This is an emergency.",Account Access,Account Access,True
908,This is unacceptable. I cannot find my invoice for order 87942.,Billing and Payments,Billing and Payments,True
446,I am really frustrated. Can I get a refund for the duplicate purchase?,Refund Requests,Refund Requests,True
941,This is the wrost experience. I cannot find my invoice for order 98944.,Billing and Payments,Billing and Payments,True


In [60]:
## ACCURACY, PRECISION, RECALL & F1 - SCORE

In [35]:
acc = accuracy_score(y_test, y_pred)        # share of correct predictions
print(f"Test accuracy: {acc:.1%}   (baseline: {baseline_acc:.1%})\n")    # compare with the base line

report_text = classification_report(y_test, y_pred, labels=labels, zero_division=0)   # text report
print(report_text)

Test accuracy: 100.0%   (baseline: 23.8%)

                               precision    recall  f1-score   support

               Account Access       1.00      1.00      1.00        36
         Billing and Payments       1.00      1.00      1.00        46
Product or Service Complaints       1.00      1.00      1.00        36
              Refund Requests       1.00      1.00      1.00        44
             Technical Issues       1.00      1.00      1.00        31

                     accuracy                           1.00       193
                    macro avg       1.00      1.00      1.00       193
                 weighted avg       1.00      1.00      1.00       193



In [62]:
## CHART of precision, recall, F1 per category

In [36]:
report = classification_report(y_test, y_pred, labels=labels,
                               output_dict=True, zero_division=0)      # the same report, as a dictionary
report_df = pd.DataFrame(report).T.loc[labels, ["precision", "recall", "f1-score"]]   # keep only the category rows
report_df = report_df.reset_index().rename(columns={"index": "category"})   # make category a normal column
report_long = report_df.melt(id_vars="category", var_name="metric", value_name="score")   # long format for plotting

px.bar(report_long, x="category", y="score", color="metric", barmode="group",   # grouped bar chart
       range_y=[0, 1.05], title="Precision, recall and F1-score per category").show()   # fixed 0-1 scale

In [64]:
## Confusion Matrix

In [37]:
cm = confusion_matrix(y_test, y_pred, labels=labels)    #count of (actual, predicted) parts

fig = px.imshow(cm, x=labels, y=labels, text_auto=True,     #heatmap eith the numbers written inside
                aspect="auto",                          # stretch the cell to fill the chart
                color_continuous_scale="Blues",    # dark blue = large number
                labels=dict(x="Predicted category", y="Actual category", color="Ticket"),   #axis names
                title="Confusion matrix (test set)")   # chart title
fig.update_xaxes(tickangle=30)           # tilt long labels so they fit
fig.show()                       #draw the shart
                            

In [66]:
## Errror analysis

In [38]:
errors = results[~results["correct"]]    #keep only the wrong prediction
print(f"Misclassified tickets: {len(errors)} out of {len(results)}")   # how many mistakes
errors.head(10)                     # show up to 10 of them

Misclassified tickets: 0 out of 193


,text,actual,predicted,correct


In [68]:
## Cross - Validation

In [39]:
pipe = Pipeline([                    # chain of steps treated as one model
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english",
                              min_df=2, sublinear_tf=True)),   # step 1: text to numbers
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               random_state=42)),              # step 2: classifier
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)     # 5 folds, same category mix in each
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")   # accuracy for each fold

print("Accuracy per fold:", np.round(scores, 3))             # score of each fold
print(f"Mean: {scores.mean():.3f}  Std: {scores.std():.3f}") # average and spread


Accuracy per fold: [1.    0.974 1.    1.    1.   ]
Mean: 0.995  Std: 0.010


In [70]:
## Which words drive each category

In [40]:
for i, category in enumerate(model.classes_):           # go through each category
    top_idx = model.coef_[1].argsort()[::-1][:8]        # position of the 8 highest weights for this category
    top_terms = [feature_names[j] for j in top_idx]      # convert position to words
    print(f"{category}:")                               # category name
    print("  ", ", ".join(top_terms))                   # its strongest words

Account Access:
   invoice, invoice order, order, payment, payment showing, days, showing pending, showing
Billing and Payments:
   invoice, invoice order, order, payment, payment showing, days, showing pending, showing
Product or Service Complaints:
   invoice, invoice order, order, payment, payment showing, days, showing pending, showing
Refund Requests:
   invoice, invoice order, order, payment, payment showing, days, showing pending, showing
Technical Issues:
   invoice, invoice order, order, payment, payment showing, days, showing pending, showing


In [72]:
# Predict a new ticket

In [41]:
def clean_text(text):                  # same cleaning function as in Module 1
    text = str(text).lower()           # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # remove web links
    text = re.sub(r"\S+@\S+", " ", text)    # remove email addresses
    text = re.sub(r"[^a-z0-9\s']", " ", text)  # remove symbols, punctuation and emojis
    return re.sub(r"\s+", " ", text).strip()   # collapse repeated spaces

def predict_ticket(text):        # predict the category of one new ticket
    cleaned = clean_text(text)   # clean it exactly like the training data
    vec = tfidf.transform([cleaned])   # convert to TF-IDF numbers
    proba = model.predict_proba(vec)[0] # probability for each category
    best = int(np.argmax(proba))        # position of the highest probability
    return model.classes_[best], float(proba[best]), {c: round(float(p), 3) for c, p in zip(model.classes_, proba)}  # label, confidence, all

label, confidence, all_probs = predict_ticket("I was charged twice and I want my money back immediately!")   # example
print(f"Predicted: {label} (confidence {confidence:.0%})")     # prediction
print("All probabilities:", all_probs)   # probabilities for every category

Predicted: Billing and Payments (confidence 42%)
All probabilities: {'Account Access': 0.133, 'Billing and Payments': 0.416, 'Product or Service Complaints': 0.146, 'Refund Requests': 0.177, 'Technical Issues': 0.128}


In [74]:
## Test on tickets written in new wording

In [42]:
new_tickets = [                                             # (ticket text, expected category)
    ("My login keeps failing even after I reset the password", "Account Access"),
    ("You took money from my card two times for the same plan", "Billing and Payments"),
    ("The mobile application closes by itself when I try to pay", "Technical Issues"),
    ("Send my money back, I am not using this service anymore", "Refund Requests"),
    ("The parcel came broken and the support staff were unhelpful", "Product or Service Complaints"),
    ("I cannot sign in and my card was also charged wrongly", "Account Access"),     # mixed ticket: ambiguous
    ("Website is down", "Technical Issues"),                                          # very short ticket
    ("Not happy", "Product or Service Complaints"),                                   # almost no information
]

rows = []                                                   # results will be collected here
for text, expected in new_tickets:                          # go through each new ticket
    pred, conf, _ = predict_ticket(text)                    # predict its category
    rows.append({"ticket": text, "expected": expected,      # store the ticket and the expected category
                 "predicted": pred, "confidence": round(conf, 2),   # the prediction and its confidence
                 "match": pred == expected})                # True when prediction equals expectation

new_results = pd.DataFrame(rows)                            # convert to a table
print(f"Correct: {new_results['match'].sum()} out of {len(new_results)}")   # how many matched
new_results                                                 # display the table

Correct: 5 out of 8


,ticket,expected,predicted,confidence,match
0,My login keeps failing even after I reset the password,Account Access,Account Access,0.64,True
1,You took money from my card two times for the same plan,Billing and Payments,Refund Requests,0.31,False
2,The mobile application closes by itself when I try to pay,Technical Issues,Product or Service Complaints,0.22,False
3,"Send my money back, I am not using this service anymore",Refund Requests,Refund Requests,0.30,True
4,The parcel came broken and the support staff were unhelpful,Product or Service Complaints,Product or Service Complaints,0.22,True
5,I cannot sign in and my card was also charged wrongly,Account Access,Billing and Payments,0.39,False
6,Website is down,Technical Issues,Technical Issues,0.32,True
7,Not happy,Product or Service Complaints,Product or Service Complaints,0.22,True


In [76]:
## Save the trained model

In [43]:
final_model = Pipeline([("tfidf", tfidf), ("clf", model)])  # join the two already-fitted steps into one object
joblib.dump(final_model, MODEL_DIR / "ticket_classifier.joblib")   # save the model to disk

metrics = {                                                 # evaluation numbers to keep for the report and the app
    "accuracy": float(acc),                                 # test accuracy
    "baseline_accuracy": float(baseline_acc),               # baseline accuracy
    "cv_mean": float(scores.mean()),                        # cross-validation mean
    "cv_std": float(scores.std()),                          # cross-validation spread
    "labels": labels,                                       # category names
    "confusion_matrix": cm.tolist(),                        # confusion matrix as a list
    "report": report,                                       # precision, recall, F1 per category
    "n_train": int(len(train_df)),                          # training set size
    "n_test": int(len(test_df)),                            # test set size
}
import json                                                 # standard library for JSON files
(MODEL_DIR / "classifier_metrics.json").write_text(json.dumps(metrics, indent=2))   # save metrics

# Check that the saved model loads and predicts the same way
loaded = joblib.load(MODEL_DIR / "ticket_classifier.joblib")   # load the model back
print("Saved and reloaded OK:", loaded.predict(["i forgot my password and cannot log in"])[0])   # quick test

Saved and reloaded OK: Account Access
